# Monitors & Reports

This file sets up the model, data, and infrastructure monitors. It also establishes a monitoring dashboard the code for generating reports on SageMaker.

Attribution: The code was made with the assistance of Perplexity.ai accessed in February 2026.

In [9]:
#Import key libraries
import sagemaker
from sagemaker import Session
from sagemaker.model_monitor import (
    DefaultModelMonitor,
    ModelQualityMonitor,
    DatasetFormat,
)
import boto3
import json
import time
import tarfile
import botocore.exceptions
import io


In [10]:
#Set up session
region = "us-east-1"
session = Session()
sm_client = boto3.client("sagemaker", region_name=region)
cw_client = boto3.client("cloudwatch", region_name=region)

role = sagemaker.get_execution_role()  # or hard‑code your SageMaker role ARN
bucket = "sagemaker-us-east-1-418418308994"
prefix = "models/benchmarks"

lr_model_artifact = "s3://sagemaker-us-east-1-418418308994/models/benchmarks/logistic_regression/model.pkl"
xgb_model_artifact = "s3://sagemaker-us-east-1-418418308994/models/benchmarks/xgboost/model.pkl"


In [12]:
#Model monitor requires .tar, repackage .pckl
def package_model_pkl(model_s3_uri, target_key):
    bucket_name = model_s3_uri.replace("s3://", "").split("/")[0]
    pkl_key = model_s3_uri.replace("s3://", "").split("/", 1)[1]
    
    s3 = boto3.client('s3')
    try:
        s3.head_object(Bucket=bucket_name, Key=pkl_key)
    except NoSuchKey:
        print(f"{model_s3_uri} not found")
        return
    
    obj = s3.get_object(Bucket=bucket_name, Key=pkl_key)
    pkl_data = obj['Body'].read()
    
    tar_buffer = io.BytesIO()
    with tarfile.open(fileobj=tar_buffer, mode='w:gz') as tar:
        tarinfo = tarfile.TarInfo('model.pkl')
        tarinfo.size = len(pkl_data)
        tar.addfile(tarinfo, io.BytesIO(pkl_data))
    
    tar_buffer.seek(0)
    tar_key = pkl_key.replace('.pkl', '.tar.gz')
    s3.put_object(Bucket=bucket_name, Key=tar_key, Body=tar_buffer)
    print(f"Packaged: s3://{bucket_name}/{tar_key}")

# XGBoost
xgb_model_artifact = "s3://sagemaker-us-east-1-418418308994/models/benchmarks/xgboost/model.pkl"
package_model_pkl(xgb_model_artifact, xgb_model_artifact)

Packaged: s3://sagemaker-us-east-1-418418308994/models/benchmarks/xgboost/model.tar.gz


In [15]:
xgb_model_artifact = "s3://sagemaker-us-east-1-418418308994/models/benchmarks/xgboost/model.tar.gz"

from sagemaker.model import Model

xgb_endpoint_name = "xgb-benchmark-endpoint"

xg_model = Model(
    image_uri="683313688378.dkr.ecr.us-east-1.amazonaws.com/sagemaker-xgboost:0.90-1-cpu-py3",
    model_data=xgb_model_artifact,
    role=role,
    sagemaker_session=session,
)

data_capture_prefix = f"{prefix}/data-capture"
data_capture_s3_uri = f"s3://{bucket}/{data_capture_prefix}"

xg_predictor = xg_model.deploy(
    initial_instance_count=1,
    instance_type="ml.m5.large",
    endpoint_name=xgb_endpoint_name,
    data_capture_config=sagemaker.model_monitor.DataCaptureConfig(
        enable_capture=True,
        sampling_percentage=100,
        destination_s3_uri=data_capture_s3_uri,
        capture_options=["REQUEST", "RESPONSE"],
    ),
)
print(f"Endpoint deploying: {xgb_endpoint_name}")

------!Endpoint deploying: xgb-benchmark-endpoint


In [17]:
baseline_data_uri = "s3://sagemaker-us-east-1-418418308994/models/benchmarks/baseline_normalized.csv"
baseline_results_uri = f"s3://{bucket}/{prefix}/monitoring/baseline"

dq_monitor = DefaultModelMonitor(
    role=role, instance_count=1, instance_type="ml.m5.xlarge",
    volume_size_in_gb=20, max_runtime_in_seconds=3600, sagemaker_session=session,
)

# Smaller sample = faster (first 2k rows)
dq_monitor.suggest_baseline(
    baseline_dataset=baseline_data_uri,
    dataset_format=DatasetFormat.csv(header=True),
    output_s3_uri=baseline_results_uri,
    wait=True,
    logs=False, #No verbose logs
)

INFO:sagemaker.image_uris:Ignoring unnecessary instance type: None.
INFO:sagemaker:Creating processing-job with name baseline-suggestion-job-2026-02-03-06-05-58-775


............................................................!

In [18]:
try:
    dq_stats_uri = dq_monitor.latest_baselining_job.baseline_statistics.file_name
    dq_constraints_uri = dq_monitor.latest_baselining_job.suggested_constraints.file_name
    print(f"Stats: {dq_stats_uri}")
    print(f"Constraints: {dq_constraints_uri}")
except:
    # List S3 output manually if SDK fails
    import boto3
    s3 = boto3.client('s3')
    response = s3.list_objects_v2(Bucket=bucket, Prefix=f"{prefix}/monitoring/baseline/")
    for obj in response.get('Contents', []):
        if obj['Key'].endswith('statistics.json'):
            dq_stats_uri = f"s3://{bucket}/{obj['Key']}"
        if obj['Key'].endswith('constraints.json'):
            dq_constraints_uri = f"s3://{bucket}/{obj['Key']}"
    print(f"Stats: {dq_stats_uri}")
    print(f"Constraints: {dq_constraints_uri}")

✅ Stats: s3://sagemaker-us-east-1-418418308994/models/benchmarks/monitoring/baseline/statistics.json
✅ Constraints: s3://sagemaker-us-east-1-418418308994/models/benchmarks/monitoring/baseline/constraints.json


### Schedule a data quality monitoring job for each endpoint

In [20]:
from sagemaker.model_monitor import CronExpressionGenerator

from sagemaker.model_monitor import CronExpressionGenerator

schedule_name_xgb_dq = "xgb-data-quality-schedule"

dq_monitor.create_monitoring_schedule(
    monitor_schedule_name=schedule_name_xgb_dq,
    endpoint_input=xgb_endpoint_name,
    output_s3_uri=f"s3://{bucket}/{prefix}/monitoring/data-quality/xgb",
    statistics="s3://sagemaker-us-east-1-418418308994/models/benchmarks/monitoring/baseline/statistics.json",
    constraints="s3://sagemaker-us-east-1-418418308994/models/benchmarks/monitoring/baseline/constraints.json",
    schedule_cron_expression=CronExpressionGenerator.hourly(),
    enable_cloudwatch_metrics=True,
)

print(f"Data quality schedule created: {schedule_name_xgb_dq}")


INFO:sagemaker.model_monitor.model_monitoring:Creating Monitoring Schedule with name: xgb-data-quality-schedule


Data quality schedule created: xgb-data-quality-schedule


In [28]:
from sagemaker.predictor import Predictor
from sagemaker.serializers import IdentitySerializer
from sagemaker.deserializers import JSONDeserializer

xg_predictor = Predictor(
    endpoint_name=xgb_endpoint_name,
    sagemaker_session=session,
    serializer=IdentitySerializer(content_type="application/json"),
    deserializer=JSONDeserializer(),
)

baseline_df = pd.read_csv("s3://sagemaker-us-east-1-418418308994/models/benchmarks/baseline_normalized.csv")
test_payload = baseline_df.drop("target", axis=1).iloc[:5].to_json(orient="records")

response = xg_predictor.predict(test_payload)
print(response)

╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <module>:15                                                                                   │
│                                                                                                  │
│   12 baseline_df = pd.read_csv("s3://sagemaker-us-east-1-418418308994/models/benchmarks/basel    │
│   13 test_payload = baseline_df.drop("target", axis=1).iloc[:5].to_json(orient="records")        │
│   14                                                                                             │
│ ❱ 15 response = xg_predictor.predict(test_payload)                                               │
│   16 print(response)                                                                             │
│   17                                                                                             │
│                                                                                                  │
│ /opt/conda/lib/python3.12/site-packages/sagemaker/base_predictor.py:212 in predict               │
│                                                                                                  │
│   209 │   │   if inference_component_name:                                                       │
│   210 │   │   │   request_args["InferenceComponentName"] = inference_component_name              │
│   211 │   │                                                                                      │
│ ❱ 212 │   │   response = self.sagemaker_session.sagemaker_runtime_client.invoke_endpoint(**req   │
│   213 │   │   return self._handle_response(response)                                             │
│   214 │                                                                                          │
│   215 │   def _handle_response(self, response):                                                  │
│                                                                                                  │
│ /opt/conda/lib/python3.12/site-packages/botocore/client.py:569 in _api_call                      │
│                                                                                                  │
│    566 │   │   │   │   │   f"{py_operation_name}() only accepts keyword arguments."              │
│    567 │   │   │   │   )                                                                         │
│    568 │   │   │   # The "self" in this scope is referring to the BaseClient.                    │
│ ❱  569 │   │   │   return self._make_api_call(operation_name, kwargs)                            │
│    570 │   │                                                                                     │
│    571 │   │   _api_call.__name__ = str(py_operation_name)                                       │
│    572                                                                                           │
│                                                                                                  │
│ /opt/conda/lib/python3.12/site-packages/botocore/client.py:1023 in _make_api_call                │
│                                                                                                  │
│   1020 │   │   │   │   "Code"                                                                    │
│   1021 │   │   │   )                                                                             │
│   1022 │   │   │   error_class = self.exceptions.from_code(error_code)                           │
│ ❱ 1023 │   │   │   raise error_class(parsed_response, operation_name)                            │
│   1024 │   │   else:                                                                             │
│   1025 │   │   │   return parsed_response                                                        │
│   1026                                                                                           │
╰────────────────────────────────────────────────────────────

### Set up a model quality moni

In [25]:
#More imports
from sagemaker.model_monitor import ModelQualityMonitor, DatasetFormat, CronExpressionGenerator
import json
from datetime import datetime, timedelta
import numpy as np

In [26]:
mq_monitor = ModelQualityMonitor(
    role=role, instance_count=1, instance_type="ml.m5.xlarge",
    volume_size_in_gb=20, max_runtime_in_seconds=3600, sagemaker_session=session,
)

model_quality_baseline_uri = f"s3://{bucket}/{prefix}/monitoring/mq-baseline"

INFO:sagemaker.image_uris:Ignoring unnecessary instance type: None.


In [27]:
# Get predictions (batch all 10k rows)
test_payload = baseline_df.drop('target', axis=1).to_json(orient='records')
predictions = xg_predictor.predict(test_payload)

# Parse XGBoost response: {'predictions': [{'score': [0.1, 0.8, 0.1]}, ...]}
prob_matrix = np.array([json.loads(p)['score'] for p in predictions['predictions']])
pred_labels = np.argmax(prob_matrix, axis=1)

# Reorder: [probs(list), pred_label(int), target(int)]
mq_baseline_df = pd.DataFrame({
    'probability': prob_matrix.tolist(),
    'predicted_label': pred_labels,
    'target': baseline_df['target'].values
})

# Save
mq_baseline_key = "models/benchmarks/mq_baseline.csv"
csv_buffer = StringIO()
mq_baseline_df.to_csv(csv_buffer, index=False)
s3_resource.Object(bucket, mq_baseline_key).put(Body=csv_buffer.getvalue())
mq_baseline_uri = f"s3://{bucket}/{mq_baseline_key}"
print(f"MQ baseline: {mq_baseline_uri} ({len(mq_baseline_df)} rows)")

╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <module>:3                                                                                    │
│                                                                                                  │
│    1 # Get predictions (batch all 10k rows)                                                      │
│    2 test_payload = baseline_df.drop('target', axis=1).to_json(orient='records')                 │
│ ❱  3 predictions = xg_predictor.predict(test_payload)                                            │
│    4                                                                                             │
│    5 # Parse XGBoost response: {'predictions': [{'score': [0.1, 0.8, 0.1]}, ...]}                │
│    6 prob_matrix = np.array([json.loads(p)['score'] for p in predictions['predictions']])        │
│                                                                                                  │
│ /opt/conda/lib/python3.12/site-packages/sagemaker/base_predictor.py:199 in predict               │
│                                                                                                  │
│   196 │   │   │   │   as is.                                                                     │
│   197 │   │   """                                                                                │
│   198 │   │   # [TODO]: clean up component_name in _create_request_args                          │
│ ❱ 199 │   │   request_args = self._create_request_args(                                          │
│   200 │   │   │   data=data,                                                                     │
│   201 │   │   │   initial_args=initial_args,                                                     │
│   202 │   │   │   target_model=target_model,                                                     │
│                                                                                                  │
│ /opt/conda/lib/python3.12/site-packages/sagemaker/base_predictor.py:259 in _create_request_args  │
│                                                                                                  │
│   256 │   │   │   else:                                                                          │
│   257 │   │   │   │   args["ContentType"] = (                                                    │
│   258 │   │   │   │   │   self.content_type                                                      │
│ ❱ 259 │   │   │   │   │   if isinstance(self.content_type, str)                                  │
│   260 │   │   │   │   │   else ", ".join(self.content_type)                                      │
│   261 │   │   │   │   )                                                                          │
│   262                                                                                            │
│                                                                                                  │
│ /opt/conda/lib/python3.12/site-packages/sagemaker/base_predictor.py:940 in content_type          │
│                                                                                                  │
│   937 │   @property                                                                              │
│   938 │   def content_type(self):                                                                │
│   939 │   │   """The MIME type of the data sent to the inference endpoint."""                    │
│ ❱ 940 │   │   return self._content_type or self.serializer.CONTENT_TYPE                          │
│   941 │                                                                                          │
│   942 │   @property                                                                              │
│   943 │   def accept(self):                                                                      │
╰────────────────────────────────────────────────────────────

In [ ]:
mq_monitor.suggest_baseline(
    baseline_dataset=mq_baseline_uri,
    dataset_format=DatasetFormat.csv(header=True),
    problem_type="MulticlassClassification",
    inference_attribute="predicted_label",
    probability_attribute="probability",
    ground_truth_attribute="target",
    output_s3_uri=model_quality_baseline_uri,
    wait=True,
)

mq_constraints = mq_monitor.suggested_constraints().file_name
print(f"MQ constraints: {mq_constraints}")


In [ ]:
# Fake production GT (uses your train labels)
ground_truth_key = f"ground-truth-{datetime.now().strftime('%Y%m%d-%H')}.jsonl"
fake_gt = []
for i, gt_label in enumerate(mq_baseline_df['target']):
    fake_gt.append(json.dumps({
        "groundTruthValue": int(gt_label),
        "eventVersion": "0", "eventId": f"lab-{i}",
        "eventTime": (datetime.now() - timedelta(hours=1)).isoformat()
    }))

s3_resource.Object(bucket, f"{prefix}/ground-truth/{ground_truth_key}").put(Body='\n'.join(fake_gt))

# Schedule
mq_monitor.create_monitoring_schedule(
    monitor_schedule_name="xgb-model-quality-schedule",
    endpoint_input=xgb_endpoint_name,
    ground_truth_input=f"s3://{bucket}/{prefix}/ground-truth/",
    output_s3_uri=f"s3://{bucket}/{prefix}/monitoring/model-quality/xgb",
    constraints=mq_constraints,
    schedule_cron_expression=CronExpressionGenerator.daily(),
    enable_cloudwatch_metrics=True,
)
print("Full model quality monitoring active!")

### Infrastructure Monitors

In [ ]:
endpoint_metric_dimensions = [
    {"Name": "EndpointName", "Value": lr_endpoint_name},
]

# Example: alarm on high 5xx error rate
cw_client.put_metric_alarm(
    AlarmName="LR-Endpoint-5XX-Errors-High",
    AlarmDescription="5XX error rate for LR endpoint above threshold",
    Namespace="AWS/SageMaker",
    MetricName="5xx",
    Dimensions=endpoint_metric_dimensions,
    Statistic="Sum",
    Period=60,
    EvaluationPeriods=5,
    Threshold=1.0,
    ComparisonOperator="GreaterThanOrEqualToThreshold",
    TreatMissingData="missing",
    ActionsEnabled=True,
    AlarmActions=["arn:aws:sns:us-east-1:ACCOUNT_ID:YourSnsTopic"],
)

# CloudWatch Monitoring Dashboard

In [ ]:
dashboard_name = "SageMaker-ML-Benchmarks"

dashboard_body = {
    "widgets": [
        {
            "type": "metric",
            "x": 0,
            "y": 0,
            "width": 12,
            "height": 6,
            "properties": {
                "title": "LR Endpoint – Invocations & Latency",
                "metrics": [
                    ["AWS/SageMaker", "Invocations", "EndpointName", lr_endpoint_name],
                    [".", "ModelLatency", ".", "."],
                ],
                "stacked": False,
                "stat": "Average",
                "period": 60,
                "region": region,
            },
        },
        {
            "type": "metric",
            "x": 0,
            "y": 6,
            "width": 12,
            "height": 6,
            "properties": {
                "title": "LR Data & Model Quality Violations",
                "metrics": [
                    [
                        "AWS/SageMaker",
                        "ModelQualityViolation",
                        "MonitoringSchedule",
                        "lr-model-quality-schedule",
                    ],
                    [
                        "AWS/SageMaker",
                        "DataQualityViolation",
                        "MonitoringSchedule",
                        "lr-data-quality-schedule",
                    ],
                ],
                "stat": "Sum",
                "period": 300,
                "region": region,
            },
        },
        # Add XGB widgets similarly
    ]
}

cw_client.put_dashboard(
    DashboardName=dashboard_name,
    DashboardBody=json.dumps(dashboard_body),
)


### Generate Model & Data Reports in SageMaker

In [ ]:
#Generate a bias report


In [ ]:
fs = s3fs.S3FileSystem()

latest_dq_output_prefix = f"{bucket}/{prefix}/monitoring/data-quality/lr"
dq_reports = fs.ls(latest_dq_output_prefix)
dq_reports

# Example: load latest constraint violations
violations_path = [p for p in dq_reports if p.endswith("constraint_violations.json")][-1]

with fs.open(violations_path, "r") as f:
    dq_violations = json.load(f)

dq_violations
